# S56R and T57P-dependent maturation of 1NC9 and CH235.12

Binding affinity against both antigens as a function of how many background mutations a genotype carries, split by the four states of S56R and T57P. Panel A is 1NC9 over N52D, S53R, G54R; panel B is CH235.12 over M34I, N52D, S53A, the same background sets Figures 4C and 5C stratify on. One point per genotype, replicate-averaged measured affinity, with a center bar per column.

## Setup

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Config

In [ ]:
data_dir = "../data"
fig_dir = "../figures"

antigens = ["CH505TF", "SF162"]
site_cols = [f"site_{i}" for i in range(10)]
floor = 6.0

center_statistic = "median"   # the draft legend says mean, the draft caption says median
rng_seed = 0                  # jitter only, no effect on any statistic

jitter_width = 0.26
center_bar_half_width = 0.36
marker_size = 9
point_alpha = 0.75
fig_size = (5.6, 8.0)
png_dpi = 300

count_colors = ["#f2a1b8", "#c3a8dd", "#8d63b5", "#3f2a5e"]

bnabs = {
    "1NC9": {"panel": "A", "focal": ["S56R", "T57P"], "background": ["N52D", "S53R", "G54R"]},
    "CH235.12": {"panel": "B", "focal": ["S56R", "T57P"], "background": ["M34I", "N52D", "S53A"]},
}

# display name to file stem; the hyphens and the dot do not appear in any filename
name_to_stem = {"1NC9": "1NC9", "CH235.12": "CH235_12", "N49P7": "N49P7",
                "NIH45-46": "NIH4546", "VRC-PG20": "VRCPG20", "VRC01": "VRC01",
                "VRC07": "VRC07", "VRC13": "VRC13", "VRC16": "VRC16"}

plt.rcParams["svg.fonttype"] = "path"   # illustrator drops live text, see the project export note
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.size"] = 10

os.makedirs(fig_dir, exist_ok=True)

## Loading

In [ ]:
def mutation_to_site(bnab):
    """Map a label like S56R onto the site_N index that the kds columns use."""
    meta = pd.read_csv(f"{data_dir}/{name_to_stem[bnab]}_meta.csv")
    return {f"{r.germline_aa}{r.kabat_position}{r.mature_aa}": int(r.site.split("_")[1])
            for r in meta.itertuples()}


def load_genotypes(bnab):
    """Collapse the two replicates to one measured affinity per genotype and antigen."""
    kds = pd.read_csv(f"{data_dir}/{name_to_stem[bnab]}_kds.csv")
    kds["tok"] = kds[site_cols].agg("".join, axis=1)
    return kds.groupby(["tok", "antigen"], as_index=False)["nlog10_Kd"].mean()


def stratify(bnab):
    """Tag every genotype with its two focal states and its background mutation count."""
    sites = mutation_to_site(bnab)
    cfg = bnabs[bnab]
    i_first, i_second = (sites[m] for m in cfg["focal"])
    i_bg = [sites[m] for m in cfg["background"]]
    df = load_genotypes(bnab)
    df["has_first"] = df["tok"].str[i_first] == "M"
    df["has_second"] = df["tok"].str[i_second] == "M"
    df["n_background"] = df["tok"].apply(lambda t: sum(t[i] == "M" for i in i_bg))
    return df


def row_states(bnab):
    """Four focal combinations in the row order Figures 4C and 5C use."""
    first, second = bnabs[bnab]["focal"]
    minus = "\u2212"
    return [(False, False, f"{minus}{first} {minus}{second}"),
            (True, False, f"+{first} {minus}{second}"),
            (False, True, f"{minus}{first} +{second}"),
            (True, True, f"+{first} +{second}")]

## Drawing

In [ ]:
def draw_cell(ax, sub, ylim, rng, show_x):
    """One focal state by antigen cell: jittered genotypes by background count, plus a center bar."""
    for count in range(4):
        vals = sub.loc[sub["n_background"] == count, "nlog10_Kd"].to_numpy(float)
        if not len(vals):
            continue
        x = count + rng.uniform(-jitter_width, jitter_width, size=len(vals))
        ax.scatter(x, vals, s=marker_size, color=count_colors[count],
                   linewidths=0, alpha=point_alpha, zorder=2)
        center = np.median(vals) if center_statistic == "median" else vals.mean()
        ax.plot([count - center_bar_half_width, count + center_bar_half_width], [center, center],
                color="black", lw=2.0, solid_capstyle="butt", zorder=3)
    ax.set_xlim(-0.6, 3.6)
    ax.set_ylim(ylim)
    ax.set_xticks(range(4))
    if not show_x:
        ax.set_xticklabels([])
    ax.tick_params(direction="out", length=3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def save_figure(fig, stem):
    """Write png, pdf and svg, then check the svg has no live text."""
    path = os.path.join(fig_dir, stem)
    fig.savefig(f"{path}.png", dpi=png_dpi, bbox_inches="tight")
    fig.savefig(f"{path}.pdf", bbox_inches="tight")
    fig.savefig(f"{path}.svg", bbox_inches="tight")
    svg = open(f"{path}.svg").read()
    assert svg.count("<text") == 0, "live text in svg, check svg.fonttype"
    print("wrote", stem, "png pdf svg")


def build_panel(bnab, out_stem=None):
    """Four focal states by two antigens for one antibody."""
    df = stratify(bnab)
    ylim = (floor - 0.15, np.nanmax(df["nlog10_Kd"]) + 0.25)   # shared so the four rows compare directly
    rng = np.random.default_rng(rng_seed)
    cfg = bnabs[bnab]

    fig, axes = plt.subplots(4, 2, figsize=fig_size, sharey=True)
    for r, (first, second, label) in enumerate(row_states(bnab)):
        for c, antigen in enumerate(antigens):
            sub = df[(df.has_first == first) & (df.has_second == second) & (df.antigen == antigen)]
            draw_cell(axes[r, c], sub, ylim, rng, show_x=(r == 3))
            if r == 0:
                axes[r, c].set_title(antigen, fontsize=11)
        axes[r, 0].set_ylabel(label, fontsize=10)

    fig.supylabel(r"Measured Binding Affinity ($-\log_{10}\mathrm{K_D}$)", fontsize=11, x=-0.02)
    fig.supxlabel("number of background mutations", fontsize=11, y=0.03)
    fig.text(0.0, 1.06, cfg["panel"], fontsize=15, fontweight="bold", va="top")

    keys = [plt.Line2D([0], [0], marker="o", linestyle="none", markersize=5,
                       markerfacecolor=count_colors[i], markeredgewidth=0, label=str(i))
            for i in range(4)]
    keys.append(plt.Line2D([0], [0], color="black", lw=2.0, label=center_statistic))
    fig.legend(handles=keys, loc="upper center", ncol=5, frameon=False,
               bbox_to_anchor=(0.5, 1.03), handletextpad=0.4, columnspacing=1.2,
               title="background mutations (" + ", ".join(cfg["background"]) + ")",
               title_fontsize=9, fontsize=9)

    fig.subplots_adjust(hspace=0.18, wspace=0.08, top=0.93, bottom=0.07, left=0.14, right=0.98)
    if out_stem:
        save_figure(fig, out_stem)
    return fig

## Build

In [ ]:
build_panel("1NC9", "fig_s6a_1NC9_focal_state_by_background_count")
plt.show()

In [ ]:
build_panel("CH235.12", "fig_s6b_CH235_12_focal_state_by_background_count")
plt.show()

## Group sizes for the caption

In [ ]:
for bnab in bnabs:
    sizes = (stratify(bnab)
             .groupby(["has_first", "has_second", "antigen", "n_background"])
             .size().rename("n").reset_index())
    print(bnab)
    display(sizes.pivot_table(index=["has_first", "has_second", "antigen"],
                              columns="n_background", values="n", fill_value=0))